In [28]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from tqdm import tqdm

# PPO-RHLF training plots

In [29]:
ENV_KEY = "pendulum"  # change to: cartpole | pendulum
ENV_ID_MAP = {"cartpole": "CartPole-v1", "pendulum": "Pendulum-v1"}
ENV_ID = ENV_ID_MAP.get(ENV_KEY, ENV_KEY)
temp_env = gym.make(ENV_ID)
ENV_CONTINUOUS = isinstance(temp_env.action_space, gym.spaces.Box)
ENV_OBS_DIM = temp_env.observation_space.shape[0]
ENV_ACTION_DIM = temp_env.action_space.shape[0] if ENV_CONTINUOUS else temp_env.action_space.n

DATASET_ROOT = Path("outputs/datasets")
OUTPUT_DIR = Path("outputs/part3")
CHECKPOINT_DIR = Path("outputs/checkpoints")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EVAL_EPISODES = 20

In [30]:
# ─── Definitions required by the plot cell ───────────────────────────────────

class RewardModel(nn.Module):
    def __init__(self, obs_dim, action_dim, continuous, hidden=64):
        super().__init__()
        self.continuous  = continuous
        self.action_dim  = action_dim
        self.net = nn.Sequential(
            nn.Linear(obs_dim + action_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),               nn.ReLU(),
            nn.Linear(hidden, 1),
        )
        self.register_buffer("bias", torch.zeros(1))

    def forward(self, obs, act):
        act_feat = act if self.continuous else F.one_hot(act, self.action_dim).float()
        return self.net(torch.cat([obs, act_feat], dim=-1)).squeeze(-1) - self.bias


def make_env(env_id, seed):
    env = gym.make(env_id)
    env.reset(seed=seed)
    return Monitor(env)


def eval_policy_true_reward(model, env_id, episodes, eval_seed):
    rng = np.random.default_rng(eval_seed)
    env = DummyVecEnv([
        lambda: make_env(env_id, seed=int(rng.integers(0, 1_000_000)))
        for _ in range(episodes)
    ])
    mean, std = evaluate_policy(model, env, n_eval_episodes=episodes, deterministic=True)
    env.close()
    return float(mean), float(std)


def evaluate_rm_signal(reward_model, env_id, n_episodes=20, seed=99):
    env = gym.make(env_id)
    rng = np.random.default_rng(seed)
    data = {n: {"true": [], "pred": [], "pred_steps": []} for n in ["pi1", "pi2"]}
    for policy, name in [(p1_policy, "pi1"), (p2_policy, "pi2")]:
        for _ in range(n_episodes):
            obs, _ = env.reset(seed=int(rng.integers(0, 1_000_000)))
            true_sum, pred_sum, done = 0.0, 0.0, False
            while not done:
                action, _ = policy.predict(obs, deterministic=True)
                obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
                if ENV_CONTINUOUS:
                    act_t = torch.tensor(np.asarray(action), dtype=torch.float32,
                                         device=DEVICE).unsqueeze(0)
                else:
                    act_t = torch.tensor(np.asarray(action), dtype=torch.long,
                                         device=DEVICE).unsqueeze(0)
                with torch.no_grad():
                    pred_r = float(reward_model(obs_t, act_t).cpu().item())
                obs, true_r, terminated, truncated, _ = env.step(action)
                true_sum += float(true_r)
                pred_sum += pred_r
                data[name]["pred_steps"].append(pred_r)
                done = terminated or truncated
            data[name]["true"].append(true_sum)
            data[name]["pred"].append(pred_sum)
    env.close()
    return data

In [32]:
# === Publication-quality PNG plots (no retraining needed) ===
# Loads all artefacts from disk. Only RM signal quality runs env rollouts (~30 s).

import pickle
import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

matplotlib.rcParams.update({
    "font.family":      "sans-serif",
    "font.sans-serif":  ["DejaVu Sans", "Arial", "Helvetica"],
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
})

# ─── Helpers ─────────────────────────────────────────────────────────────────────────────
def _style(ax, xlabel='', ylabel=''):
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(linewidth=0.5, linestyle="-")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)

def _legend(ax, **kw):
    """Opaque white box legend, consistent style across all figures."""
    kw.setdefault("markerscale", 1.1)   # caller can override without conflict
    leg = ax.legend(
        fontsize=8, frameon=True, facecolor='white',
        framealpha=1.0, edgecolor='0.6',
        **kw
    )
    leg.get_frame().set_linewidth(0.8)
    return leg

def _save(fig, name):
    p = png_dir / name
    fig.savefig(p, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  saved {p.name}")

def _load(path):
    with open(path, "rb") as f:
        return pickle.load(f)

# ─── Load artefacts from disk ────────────────────────────────────────────────────────
_rmdir = OUTPUT_DIR / "reward_models"

if "rm_histories" not in globals() or not rm_histories:
    rm_histories = _load(_rmdir / f"rm_histories_{ENV_KEY}.pkl")
    print(f"Loaded rm_histories ({len(rm_histories)} entries)")
if "rm_meta" not in globals() or not rm_meta:
    rm_meta = _load(_rmdir / f"rm_meta_{ENV_KEY}.pkl")
if "rm_models" not in globals() or not rm_models:
    rm_models = {}
    for _p in sorted(_rmdir.glob(f"{ENV_KEY}_*.pt")):
        _s = torch.load(_p, map_location=DEVICE)
        _h = _s["net.0.weight"].shape[0]
        _m = RewardModel(ENV_OBS_DIM, ENV_ACTION_DIM, ENV_CONTINUOUS, hidden=_h).to(DEVICE)
        _m.load_state_dict(_s)
        _m.eval()
        rm_models[_p.stem] = _m
    print(f"Loaded {len(rm_models)} reward models")
if "dpo_histories" not in globals() or not dpo_histories:
    dpo_histories = _load(OUTPUT_DIR / f"dpo_histories_{ENV_KEY}.pkl")
    print(f"Loaded dpo_histories ({len(dpo_histories)} entries)")
if "ppo_histories" not in globals() or not ppo_histories:
    ppo_histories = _load(OUTPUT_DIR / f"ppo_histories_{ENV_KEY}.pkl")
    print(f"Loaded ppo_histories ({len(ppo_histories)} entries)")
if "df" not in globals() or df.empty:
    df = pd.read_csv(OUTPUT_DIR / f"results_{ENV_KEY}.csv")
    print(f"Loaded results ({len(df)} rows)")
if "p1_policy" not in globals() or p1_policy is None:
    p1_policy = PPO.load(CHECKPOINT_DIR / f"pi1_{ENV_KEY}_seed0.zip", device=DEVICE).policy
if "p2_policy" not in globals() or p2_policy is None:
    p2_policy = PPO.load(CHECKPOINT_DIR / f"pi2_{ENV_KEY}_seed0.zip", device=DEVICE).policy

png_dir = OUTPUT_DIR / "plots_png"
png_dir.mkdir(parents=True, exist_ok=True)

# ─── Baselines (quick eval — no training) ─────────────────────────────────────────────
print("Evaluating π1 / π2 baselines...")
pi1_mean, pi1_std = eval_policy_true_reward(p1_policy, ENV_ID, EVAL_EPISODES, eval_seed=42)
pi2_mean, pi2_std = eval_policy_true_reward(p2_policy, ENV_ID, EVAL_EPISODES, eval_seed=42)
print(f"  π1: {pi1_mean:.1f} ± {pi1_std:.1f}   π2: {pi2_mean:.1f} ± {pi2_std:.1f}")

# ─── RM signal rollouts (evaluation only) ────────────────────────────────────────────
print("Running RM signal evaluation rollouts...")
rm_names_sorted = sorted(rm_meta.keys(), key=lambda n: rm_meta[n].get("K", 0))
rm_signal_data = {}
for rm_name in rm_names_sorted:
    k  = rm_meta[rm_name].get("K", rm_name)
    rm = rm_models.get(rm_name)
    if rm is None:
        print(f"  K={k} model not loaded — skipping")
        continue
    print(f"  K={k} ...", end=" ", flush=True)
    rm_signal_data[k] = evaluate_rm_signal(rm, ENV_ID)
    print("done")

# K → colour from matplotlib default cycle (C0–C9)
all_ks = sorted(set(dpo_histories) | set(ppo_histories) | set(rm_signal_data))
k_clr  = {k: f"C{i % 10}" for i, k in enumerate(all_ks)}

print("\nGenerating PNG plots...")

# ══════════════════════════════════════════════════════════════════════════════
# Figure 1 — Performance vs K
# ══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(8, 4))
for algo, mean_col, std_col in [
    ("DPO",      "dpo_return", "dpo_std"),
    ("PPO-RLHF", "ppo_return", "ppo_std"),
]:
    sub = df.groupby('K').agg({mean_col: 'mean', std_col: 'mean'}).reset_index()
    ax.plot(sub['K'], sub[mean_col], marker='o', label=algo)
    # std band: capped at ±400 so wide bands don't drown the lines
    lower = np.maximum(sub[mean_col] - sub[std_col], sub[mean_col] - 400)
    upper = np.minimum(sub[mean_col] + sub[std_col], sub[mean_col] + 400)
    ax.fill_between(sub['K'], lower, upper, alpha=0.2)

# # π₁ / π₂ reference baselines
# ax.axhline(pi1_mean, color="C4", lw=1.0, ls="--", label=f"π₁ ({pi1_mean:.0f})")
# ax.axhline(pi2_mean, color="C5", lw=1.0, ls=":", label=f"π₂ ({pi2_mean:.0f})")

# Log x-axis with ticks at the actual K values
ax.set_xscale("log")
_ks = sorted(df['K'].unique())
ax.set_xticks(_ks)
ax.set_xticklabels([f"{k}" if k < 1000 else f"{k // 1000}k" for k in _ks])
ax.xaxis.set_minor_locator(mticker.NullLocator())   # suppress minor log ticks

_style(ax, xlabel='K (preference dataset size)', ylabel='Average return')
_legend(ax, loc="lower right")
plt.tight_layout(pad=0.8)
_save(fig, "performance_vs_K.png")

# ══════════════════════════════════════════════════════════════════════════════
# Figure 2 — Checkpoint reward curves  (DPO | PPO-RLHF, shared y-axis)
# ══════════════════════════════════════════════════════════════════════════════
YLIM     = {'pendulum': (-1800, 50), 'cartpole': (-10, 520)}.get(ENV_KEY, (-1800, 50))
DPO_XLIM = {'pendulum': 2_000_000,  'cartpole': None}.get(ENV_KEY)

if DPO_XLIM:
    _dpo_xlabel = f"DPO training steps (×10⁶, first {DPO_XLIM // 1_000_000:.0f}M)"
else:
    _dpo_xlabel = "DPO training steps"

fig, (ax_dpo, ax_ppo) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, hists, xlabel, x_clip in [
    (ax_dpo, dpo_histories,  _dpo_xlabel,               DPO_XLIM),
    (ax_ppo, ppo_histories,  "PPO-RLHF training steps", None),
]:
    for k in all_ks:
        h = hists.get(k)
        if not h:
            continue
        steps = np.asarray(h["steps"])
        mr    = np.asarray(h["mean_reward"])
        sr    = np.asarray(h["std_reward"])

        if x_clip is not None:
            mask = steps <= x_clip
            steps, mr, sr = steps[mask], mr[mask], sr[mask]
        if len(steps) == 0:
            continue

        clr          = k_clr[k]
        marker_every = max(1, len(steps) // 30)   # ≤ 30 markers per curve

        ax.plot(steps, mr, color=clr, lw=1.5,
                marker="o", ms=4, markevery=marker_every, label=f"K={k}")

        # std band: capped at ±400 so wide bands don’t drown the lines
        lower = np.maximum(mr - sr, mr - 400)
        upper = np.minimum(mr + sr, mr + 400)
        ax.fill_between(steps, lower, upper, color=clr, alpha=0.08)

    ax.set_ylim(*YLIM)
    if x_clip is not None:
        ax.set_xlim(0, x_clip)
    _style(ax, xlabel=xlabel, ylabel="True reward")
    _legend(ax, loc="lower right", bbox_to_anchor=(0.99, 0.01),
            borderpad=0.4, labelspacing=0.25)

ax_ppo.set_ylabel("")   # sharey — no duplicate label on right panel

# DPO x-axis: plain decimals (e.g. "0.5", "1.0") with scale already in the label
_max_dpo_step = max((max(h['steps']) for h in dpo_histories.values() if h), default=0)
if _max_dpo_step >= 500_000:
    ax_dpo.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f"{x / 1e6:.1f}")
    )

plt.tight_layout(pad=0.8)
_save(fig, "checkpoint_curves.png")

# ══════════════════════════════════════════════════════════════════════════════
# Figure 3 — RM signal: true vs predicted return scatter grid
# ══════════════════════════════════════════════════════════════════════════════
ks_sig  = sorted(rm_signal_data)
ncols_s = len(ks_sig)   # all K values in a single row
nrows_s = 1
fig, axes = plt.subplots(nrows_s, ncols_s,
                          figsize=(4.8 * ncols_s, 3.8),
                          sharex=True, sharey=True, squeeze=False)

for idx, k in enumerate(ks_sig):
    ax = axes[0][idx]
    data = rm_signal_data[k]
    ax.scatter(data["pi1"]["true"], data["pi1"]["pred"],
               color="C2", alpha=0.85, s=40, edgecolors="white",
               linewidths=0.3, label="π₁", zorder=3)
    ax.scatter(data["pi2"]["true"], data["pi2"]["pred"],
               color="C3", alpha=0.85, s=40, edgecolors="white",
               linewidths=0.3, label="π₂", zorder=3)
    ax.set_title(f"K = {k}", fontsize=10, fontweight="bold")
    _style(ax)
    ax.set_xlabel("True return", fontsize=9)
    if idx == 0:
        ax.set_ylabel("Predicted return", fontsize=9)
    _legend(ax, markerscale=1.5, loc="lower right")

plt.tight_layout(pad=0.8)
_save(fig, "rm_signal_scatter_grid.png")


Evaluating π1 / π2 baselines...
  π1: -304.2 ± 170.1   π2: -511.7 ± 290.7
Running RM signal evaluation rollouts...
  K=500 ... done
  K=1000 ... done
  K=5000 ... done
  K=10000 ... done

Generating PNG plots...
  saved performance_vs_K.png
  saved checkpoint_curves.png
  saved rm_signal_scatter_grid.png
